# Telecom X — Análisis de Evasión de Clientes (Churn)

## Objetivo

Realizar un análisis exploratorio y limpieza de datos de clientes de **Telecom X** con el fin de detectar patrones asociados a la **evasión de clientes (churn)**: aquellos usuarios que cancelan sus servicios.

## Pasos del Análisis

1. Extracción y carga de datos
2. Exploración inicial del dataset
3. Limpieza y transformación de datos
4. Análisis Exploratorio de Datos (EDA)
5. Análisis de patrones de churn
6. Conclusiones e insights

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests
import warnings

warnings.filterwarnings('ignore')

# Estilo de visualizaciones
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Librerías importadas correctamente.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  seaborn : {sns.__version__}')

## 2. Extracción y Carga de Datos

Los datos se obtienen desde una fuente JSON pública que contiene información anidada sobre clientes de Telecom X.

In [ ]:
URL_DATOS = (
    'https://raw.githubusercontent.com/ingridcristh/'
    'challenge2-data-science-LATAM/main/TelecomX_Data.json'
)

respuesta = requests.get(URL_DATOS)
respuesta.raise_for_status()

# Normalizar estructura JSON anidada a DataFrame plano
df_raw = pd.json_normalize(respuesta.json())

print(f'Datos cargados exitosamente: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas')
df_raw.head(3)

## 3. Exploración Inicial del Dataset

In [ ]:
print('=== Forma del dataset ===')
print(f'{df_raw.shape[0]} registros, {df_raw.shape[1]} columnas\n')

print('=== Columnas y tipos de datos ===')
print(df_raw.dtypes.to_string())

In [ ]:
print('=== Primeras 5 filas ===')
df_raw.head()

In [ ]:
print('=== Estadísticas descriptivas (variables numéricas) ===')
df_raw.describe()

In [ ]:
print('=== Valores nulos por columna ===')
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos})
print(resumen_nulos[resumen_nulos['Nulos'] > 0].to_string() or 'No se encontraron valores nulos.')

In [ ]:
print('=== Registros duplicados ===')
duplicados = df_raw.duplicated().sum()
print(f'{duplicados} fila(s) duplicada(s) encontrada(s).')

## 4. Limpieza y Transformación de Datos

In [ ]:
df = df_raw.copy()

# --- 4.1  Renombrar columnas para mayor legibilidad ---
df.rename(columns={
    'customerID'                  : 'cliente_id',
    'Churn'                       : 'churn',
    'customer.gender'             : 'genero',
    'customer.SeniorCitizen'      : 'adulto_mayor',
    'customer.Partner'            : 'tiene_pareja',
    'customer.Dependents'         : 'tiene_dependientes',
    'customer.tenure'             : 'meses_contrato',
    'phone.PhoneService'          : 'servicio_telefono',
    'phone.MultipleLines'         : 'lineas_multiples',
    'internet.InternetService'    : 'servicio_internet',
    'internet.OnlineSecurity'     : 'seguridad_online',
    'internet.OnlineBackup'       : 'respaldo_online',
    'internet.DeviceProtection'   : 'proteccion_dispositivo',
    'internet.TechSupport'        : 'soporte_tecnico',
    'internet.StreamingTV'        : 'streaming_tv',
    'internet.StreamingMovies'    : 'streaming_peliculas',
    'account.Contract'            : 'tipo_contrato',
    'account.PaperlessBilling'    : 'factura_digital',
    'account.PaymentMethod'       : 'metodo_pago',
    'account.Charges.Monthly'     : 'cargo_mensual',
    'account.Charges.Total'       : 'cargo_total',
}, inplace=True)

print('Columnas renombradas:', df.columns.tolist())

In [ ]:
# --- 4.2  Convertir 'cargo_total' a numérico (puede venir como string) ---
df['cargo_total'] = pd.to_numeric(df['cargo_total'], errors='coerce')

# --- 4.3  Rellenar nulos en 'cargo_total' con 0 (clientes nuevos sin cargos acumulados) ---
nulos_cargo = df['cargo_total'].isnull().sum()
print(f'Nulos en cargo_total antes de imputar: {nulos_cargo}')
df['cargo_total'].fillna(0, inplace=True)

# --- 4.4  Convertir 'churn' a variable binaria (1=Sí, 0=No) ---
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})

# --- 4.5  Eliminar filas con cliente_id nulo ---
antes = len(df)
df.dropna(subset=['cliente_id'], inplace=True)
print(f'Filas eliminadas por cliente_id nulo: {antes - len(df)}')

# --- 4.6  Eliminar duplicados ---
antes = len(df)
df.drop_duplicates(inplace=True)
print(f'Filas duplicadas eliminadas: {antes - len(df)}')

print(f'\nDataset limpio: {df.shape[0]} registros, {df.shape[1]} columnas')

In [ ]:
# --- 4.7  Verificación final de nulos ---
print('=== Verificación de nulos tras limpieza ===')
nulos_final = df.isnull().sum()
print(nulos_final[nulos_final > 0].to_string() or 'Sin valores nulos restantes. ✓')

In [ ]:
print('=== Vista del dataset limpio ===')
df.head()

## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Tasa Global de Churn

In [ ]:
tasa_churn = df['churn'].mean() * 100
conteo_churn = df['churn'].value_counts()

print(f'Tasa de evasión (churn): {tasa_churn:.2f}%')
print(f'Clientes que se quedan : {conteo_churn[0]:,}')
print(f'Clientes que se van    : {conteo_churn[1]:,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
colores = ['#2ecc71', '#e74c3c']
axes[0].bar(['Se queda', 'Se va'], conteo_churn.values, color=colores, edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribución de Churn')
axes[0].set_ylabel('Cantidad de Clientes')
for i, v in enumerate(conteo_churn.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(
    conteo_churn.values,
    labels=['Se queda', 'Se va'],
    autopct='%1.1f%%',
    colors=colores,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Proporción de Churn')

plt.suptitle('Tasa Global de Evasión de Clientes — Telecom X', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Distribución de Variables Numéricas

In [ ]:
vars_numericas = ['meses_contrato', 'cargo_mensual', 'cargo_total']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

titulos = {
    'meses_contrato' : 'Antigüedad del Contrato (meses)',
    'cargo_mensual'  : 'Cargo Mensual (USD)',
    'cargo_total'    : 'Cargo Total Acumulado (USD)',
}

for ax, col in zip(axes, vars_numericas):
    ax.hist(df[col], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='#2ecc71', linestyle='--', linewidth=1.5, label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(titulos[col])
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Distribución de Variables Categóricas Principales

In [ ]:
vars_cat = [
    ('genero', 'Género'),
    ('tipo_contrato', 'Tipo de Contrato'),
    ('metodo_pago', 'Método de Pago'),
    ('servicio_internet', 'Servicio de Internet'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, titulo) in zip(axes.flatten(), vars_cat):
    conteo = df[col].value_counts()
    ax.barh(conteo.index, conteo.values, color=sns.color_palette('muted', len(conteo)))
    ax.set_title(titulo)
    ax.set_xlabel('Cantidad de Clientes')
    for i, v in enumerate(conteo.values):
        ax.text(v + 10, i, str(v), va='center', fontsize=9)

plt.suptitle('Distribución de Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Análisis de Patrones de Churn

En esta sección comparamos el comportamiento de clientes que se van versus los que permanecen.

### 6.1 Variables Numéricas por Churn

In [ ]:
etiquetas_churn = {0: 'Se queda', 1: 'Se va'}
df['churn_label'] = df['churn'].map(etiquetas_churn)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colores_churn = {'Se queda': '#2ecc71', 'Se va': '#e74c3c'}

for ax, (col, titulo) in zip(axes, [
    ('meses_contrato', 'Antigüedad del Contrato'),
    ('cargo_mensual', 'Cargo Mensual (USD)'),
    ('cargo_total', 'Cargo Total Acumulado (USD)'),
]):
    sns.boxplot(
        data=df, x='churn_label', y=col,
        palette=colores_churn, ax=ax, order=['Se queda', 'Se va']
    )
    ax.set_title(titulo)
    ax.set_xlabel('')
    ax.set_ylabel(col)

    # Anotar mediana
    for i, grupo in enumerate(['Se queda', 'Se va']):
        mediana = df.loc[df['churn_label'] == grupo, col].median()
        ax.text(i, mediana + (ax.get_ylim()[1] * 0.02), f'{mediana:.0f}',
                ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Variables Numéricas según Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Estadísticas por grupo de Churn ===')
df.groupby('churn_label')[['meses_contrato', 'cargo_mensual', 'cargo_total']].agg(['mean', 'median']).round(2)

### 6.2 Tasa de Churn por Variables Categóricas

In [ ]:
def graficar_churn_categorico(columna, titulo, ax):
    """Grafica la tasa de churn (%) para cada categoría de la variable dada."""
    tasa = df.groupby(columna)['churn'].mean().mul(100).sort_values(ascending=False)
    colores_barra = ['#e74c3c' if v > tasa_churn else '#3498db' for v in tasa.values]
    bars = ax.barh(tasa.index, tasa.values, color=colores_barra, edgecolor='white')
    ax.axvline(tasa_churn, color='gray', linestyle='--', linewidth=1, label=f'Media global: {tasa_churn:.1f}%')
    ax.set_title(titulo)
    ax.set_xlabel('Tasa de Churn (%)')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=8)
    for bar, v in zip(bars, tasa.values):
        ax.text(v + 0.3, bar.get_y() + bar.get_height() / 2, f'{v:.1f}%', va='center', fontsize=8)


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

graficar_churn_categorico('tipo_contrato', 'Churn por Tipo de Contrato', axes[0, 0])
graficar_churn_categorico('metodo_pago', 'Churn por Método de Pago', axes[0, 1])
graficar_churn_categorico('servicio_internet', 'Churn por Servicio de Internet', axes[1, 0])
graficar_churn_categorico('genero', 'Churn por Género', axes[1, 1])

plt.suptitle('Tasa de Churn por Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Churn por Servicios Adicionales

In [ ]:
servicios = [
    'seguridad_online', 'respaldo_online', 'proteccion_dispositivo',
    'soporte_tecnico', 'streaming_tv', 'streaming_peliculas'
]

tasa_por_servicio = {}
for srv in servicios:
    temp = df[df[srv].isin(['Yes', 'No'])]
    tasa_por_servicio[srv] = temp.groupby(srv)['churn'].mean().mul(100)

tasa_df = pd.DataFrame(tasa_por_servicio).T.rename(columns={'No': 'Sin servicio', 'Yes': 'Con servicio'})

tasa_df.index = [
    'Seguridad Online', 'Respaldo Online', 'Protección Dispositivo',
    'Soporte Técnico', 'Streaming TV', 'Streaming Películas'
]

ax = tasa_df.plot(
    kind='barh', color=['#e74c3c', '#2ecc71'],
    figsize=(11, 6), edgecolor='white'
)
ax.set_title('Churn por Contratación de Servicios Adicionales', fontsize=14, fontweight='bold')
ax.set_xlabel('Tasa de Churn (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(title='Estado del Servicio')
plt.tight_layout()
plt.show()

print(tasa_df.round(2).to_string())

### 6.4 Churn según Perfil del Cliente

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, titulo) in zip(axes, [
    ('adulto_mayor', 'Adulto Mayor (0=No, 1=Sí)'),
    ('tiene_pareja', 'Tiene Pareja'),
    ('tiene_dependientes', 'Tiene Dependientes'),
]):
    tasa = df.groupby(col)['churn'].mean().mul(100)
    colores_barra = ['#e74c3c' if v > tasa_churn else '#3498db' for v in tasa.values]
    ax.bar(tasa.index.astype(str), tasa.values, color=colores_barra, edgecolor='white')
    ax.axhline(tasa_churn, color='gray', linestyle='--', linewidth=1, label=f'Media: {tasa_churn:.1f}%')
    ax.set_title(titulo)
    ax.set_ylabel('Tasa de Churn (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=8)
    for i, v in enumerate(tasa.values):
        ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn según Perfil Demográfico del Cliente', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.5 Antigüedad del Contrato y Churn

In [ ]:
# Agrupar por tramos de antigüedad
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ['0–12m', '13–24m', '25–36m', '37–48m', '49–60m', '61–72m']
df['tramo_antiguedad'] = pd.cut(df['meses_contrato'], bins=bins, labels=labels, include_lowest=True)

churn_por_tramo = df.groupby('tramo_antiguedad', observed=True)['churn'].mean().mul(100)
conteo_por_tramo = df.groupby('tramo_antiguedad', observed=True)['churn'].count()

fig, ax1 = plt.subplots(figsize=(11, 5))

color_barra = '#3498db'
ax1.bar(churn_por_tramo.index, conteo_por_tramo.values, color=color_barra, alpha=0.4, label='Total clientes')
ax1.set_xlabel('Tramo de Antigüedad')
ax1.set_ylabel('Cantidad de Clientes', color=color_barra)
ax1.tick_params(axis='y', labelcolor=color_barra)

ax2 = ax1.twinx()
ax2.plot(churn_por_tramo.index, churn_por_tramo.values, color='#e74c3c', marker='o', linewidth=2, label='Tasa de churn')
ax2.set_ylabel('Tasa de Churn (%)', color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

# Leyendas combinadas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Antigüedad del Contrato vs Tasa de Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.6 Mapa de Calor — Correlaciones

In [ ]:
# Codificar variables binarias para el mapa de calor
df_corr = df[[
    'churn', 'meses_contrato', 'cargo_mensual', 'cargo_total',
    'adulto_mayor'
]].copy()

binarias = {
    'genero'             : {'Male': 1, 'Female': 0},
    'tiene_pareja'       : {'Yes': 1, 'No': 0},
    'tiene_dependientes' : {'Yes': 1, 'No': 0},
    'servicio_telefono'  : {'Yes': 1, 'No': 0},
    'factura_digital'    : {'Yes': 1, 'No': 0},
}

for col, mapeo in binarias.items():
    df_corr[col] = df[col].map(mapeo)

corr = df_corr.corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Mapa de Calor — Correlaciones entre Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Correlaciones con Churn (ordenadas) ===')
print(corr['churn'].drop('churn').sort_values(key=abs, ascending=False).round(3).to_string())

### 6.7 Distribución de Cargo Mensual por Tipo de Contrato y Churn

In [ ]:
g = sns.FacetGrid(
    df, col='tipo_contrato', hue='churn_label',
    palette=colores_churn, height=4, aspect=1.1
)
g.map(sns.histplot, 'cargo_mensual', bins=20, alpha=0.6, kde=True)
g.add_legend(title='Estado')
g.set_axis_labels('Cargo Mensual (USD)', 'Frecuencia')
g.set_titles('Contrato: {col_name}')
g.figure.suptitle(
    'Distribución de Cargo Mensual por Tipo de Contrato y Churn',
    fontsize=13, fontweight='bold', y=1.03
)
plt.tight_layout()
plt.show()

## 7. Resumen de Hallazgos y Conclusiones

In [ ]:
print('=' * 65)
print('        RESUMEN EJECUTIVO — ANÁLISIS DE CHURN TELECOM X')
print('=' * 65)

total = len(df)
churners = df['churn'].sum()
tasa = churners / total * 100

print(f'\n📊 MÉTRICAS GENERALES')
print(f'   Total de clientes analizados : {total:,}')
print(f'   Clientes que abandonaron     : {churners:,} ({tasa:.1f}%)')
print(f'   Clientes que permanecen      : {total - churners:,} ({100 - tasa:.1f}%)')

print(f'\n🔍 PRINCIPALES FACTORES DE RIESGO')

# Tipo de contrato
churn_contrato = df.groupby('tipo_contrato')['churn'].mean().mul(100).sort_values(ascending=False)
print(f'\n   1. TIPO DE CONTRATO')
for k, v in churn_contrato.items():
    marca = '⚠️ ' if v > tasa_churn else '✅ '
    print(f'      {marca}{k:<20} → {v:.1f}% de churn')

# Servicio de internet
churn_internet = df.groupby('servicio_internet')['churn'].mean().mul(100).sort_values(ascending=False)
print(f'\n   2. SERVICIO DE INTERNET')
for k, v in churn_internet.items():
    marca = '⚠️ ' if v > tasa_churn else '✅ '
    print(f'      {marca}{k:<20} → {v:.1f}% de churn')

# Estadísticas de antigüedad
media_queda = df[df['churn'] == 0]['meses_contrato'].mean()
media_va    = df[df['churn'] == 1]['meses_contrato'].mean()
print(f'\n   3. ANTIGÜEDAD PROMEDIO')
print(f'      ✅ Clientes que se quedan : {media_queda:.1f} meses')
print(f'      ⚠️  Clientes que se van   : {media_va:.1f} meses')

# Cargo mensual
media_cargo_queda = df[df['churn'] == 0]['cargo_mensual'].mean()
media_cargo_va    = df[df['churn'] == 1]['cargo_mensual'].mean()
print(f'\n   4. CARGO MENSUAL PROMEDIO')
print(f'      ✅ Clientes que se quedan : USD {media_cargo_queda:.2f}')
print(f'      ⚠️  Clientes que se van   : USD {media_cargo_va:.2f}')

print(f'\n💡 CONCLUSIONES CLAVE')
print('   • Los contratos mes a mes tienen la mayor tasa de churn.')
print('   • Los clientes con servicio de fibra óptica presentan mayor evasión.')
print('   • Los clientes que se van tienen menor antigüedad (los primeros meses son críticos).')
print('   • Los churners pagan cargos mensuales más altos en promedio.')
print('   • Adultos mayores y clientes sin dependientes tienen mayor propensión al churn.')
print('   • Los servicios adicionales (seguridad, soporte, respaldo) reducen el churn.')
print('   • El pago electrónico (cheque electrónico) está fuertemente asociado al churn.')
print()
print('=' * 65)